# Introduction

This is the basic reproduction experiment for Yao et al. "Dynamic Word Embeddings for Evolving Semantic Discovery".
It is based on the source code provided in their
[GitHub repository](https://github.com/yifan0sun/DynamicWord2Vec)
and aims to simplify the experiment, update it to a modern Python setup, and ultimately validate the findings from the authors.

# Setup

## Imports

In [31]:
import time

# file io
from pathlib import Path
import pickle
import scipy.io as sio

# data processing
import numpy as np
import pandas as pd
import scipy.sparse as ss

# analyses and plots
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

## Data setup
The experiment relies on files from the researchers dropbox folder. These files are saved in the data directory first.

In [32]:
from setup.setup import load_data
load_data()

Data seems to be loaded already. Skipping download.


## Static variables

In [33]:
nw = 20936 # number of words in vocab (11068100/20936 for ngram/nyt)
T = range(1990,2016) # total number of time points (20/range(27) for ngram/nyt)
cuda = True

embeddings = sio.loadmat("../data/emb_static.mat")["emb"]
trainhead = "../data/wordPairPMI_"
savehead = "../results/"

Outputs will be stored in the results folder.

In [36]:
Path(savehead).mkdir(exist_ok=True)

# Embedding training

## Helper functions

In [37]:
# the equations are to update V. So:
# Y is n X b (b = batch size)
# r = rank
# U is n X r
# Vm1 and Vp1 are bXr. so they are b rows of V, transposed
def update(U, Y, Vm1, Vp1, lam, tau, gam, ind, iflag):
    UtU = np.dot(U.T, U) # rxr
    r = UtU.shape[0]
    M = UtU + (lam + 2 * tau + gam) * np.eye(r) if iflag else UtU + (lam + tau + gam) * np.eye(r)

    Uty = np.dot(U.T, Y) # rxb
    Ub = U[ind, :].T   # rxb
    A = Uty + gam * Ub + tau * (Vm1.T + Vp1.T)  # rxb
    Vhat = np.linalg.lstsq(M, A) # rxb
    return Vhat[0].T # bxr

def get_batches(vocab, b):
    batchinds = []
    current = 0
    while current < vocab:
        inds = range(current, min(current + b, vocab))
        current = min(current + b, vocab)
        batchinds.append(inds)
    return batchinds

## Training parameters

In [38]:
ITERS = 5 # total passes over the data
lam = 10 # frob regularizer
gam = 100 # forcing regularizer
tau = 50  # smoothing regularizer
r   = 50  # rank
b = nw # batch size
emph = 1 # emphasize the nonzero

savefile = savehead + 'L' + str(lam) + 'T' + str(tau) + 'G' + str(gam) + 'A' + str(emph)

f"Output file: {savefile}"

'Output file: ../results/L10T50G100A1'

## Training procedure

In [39]:
print("starting training")
print(f"there are a total of {nw} words, and {T} time points")

print('initializing')

Ulist = [embeddings.copy() for t in T]
Vlist = [embeddings.copy() for t in T]

print('getting batch indices')
if b < nw:
    b_ind = get_batches(nw, b)
else:
    b_ind = [list(range(nw))]

start_time = time.time()

# sequential updates
for iteration in range(ITERS):
    print(f"Iteration {iteration}")
    try:
        # Using context managers for pickle loading
        with open(f"{savefile}ngU_iter{iteration}.p", "rb") as f_u:
            Ulist = pickle.load(f_u)
        with open(f"{savefile}ngV_iter{iteration}.p", "rb") as f_v:
            Vlist = pickle.load(f_v)
        print(f"iteration {iteration} loaded successfully")
        continue
    except (IOError, EOFError, FileNotFoundError):
        pass

    loss = 0
    # shuffle times
    times = list(range(len(T))) if iteration == 0 else np.random.permutation(len(T))

    for t in times:
        print(f"iteration {iteration}, time {t}")
        filename = f"{trainhead}{t}.csv"
        print(filename)

        pmi = pd.read_csv(filename).values
        pmi = ss.coo_matrix((pmi[:, 2], (pmi[:, 0], pmi[:, 1])), shape=(nw, nw))

        for j, ind in enumerate(b_ind):
            print(f"{j} out of {len(b_ind)}")

            pmi_seg = pmi[:, ind].todense()

            # Update boundaries logic
            if t == 0:
                vp, up = np.zeros((len(ind), r)), np.zeros((len(ind), r))
                iflag = True
            else:
                vp, up = Vlist[t-1][ind, :], Ulist[t-1][ind, :]
                iflag = False

            if t == len(T) - 1:
                vn, un = np.zeros((len(ind), r)), np.zeros((len(ind), r))
                iflag = True
            else:
                vn, un = Vlist[t+1][ind, :], Ulist[t+1][ind, :]
                iflag = False

            # Perform updates
            Vlist[t][ind, :] = update(Ulist[t], emph * pmi_seg, vp, vn, lam, tau, gam, ind, iflag)
            Ulist[t][ind, :] = update(Vlist[t], emph * pmi_seg, up, un, lam, tau, gam, ind, iflag)

print(f"time elapsed = {time.time() - start_time}")

with open(f"{savefile}ngU_iter{iteration}.p", "wb") as f_u:
    pickle.dump(Ulist, f_u, pickle.HIGHEST_PROTOCOL)
with open(f"{savefile}ngV_iter{iteration}.p", "wb") as f_v:
    pickle.dump(Vlist, f_v, pickle.HIGHEST_PROTOCOL)

starting training
there are a total of 20936 words, and range(1990, 2016) time points
initializing
getting batch indices
Iteration 0
iteration 0, time 0
../data/wordPairPMI_0.csv
0 out of 1
iteration 0, time 1
../data/wordPairPMI_1.csv
0 out of 1
iteration 0, time 2
../data/wordPairPMI_2.csv
0 out of 1
iteration 0, time 3
../data/wordPairPMI_3.csv
0 out of 1
iteration 0, time 4
../data/wordPairPMI_4.csv
0 out of 1
iteration 0, time 5
../data/wordPairPMI_5.csv
0 out of 1
iteration 0, time 6
../data/wordPairPMI_6.csv
0 out of 1
iteration 0, time 7
../data/wordPairPMI_7.csv
0 out of 1
iteration 0, time 8
../data/wordPairPMI_8.csv
0 out of 1
iteration 0, time 9
../data/wordPairPMI_9.csv
0 out of 1
iteration 0, time 10
../data/wordPairPMI_10.csv
0 out of 1
iteration 0, time 11
../data/wordPairPMI_11.csv
0 out of 1
iteration 0, time 12
../data/wordPairPMI_12.csv
0 out of 1
iteration 0, time 13
../data/wordPairPMI_13.csv
0 out of 1
iteration 0, time 14
../data/wordPairPMI_14.csv
0 out of 1
it

# Output visualization

## Setup word to id mapping

In [41]:
with open('../data/wordlist.txt', 'r') as fid:
    wordlist = [line.strip() for line in fid]

word2Id = {word: i for i, word in enumerate(wordlist)}

times = range(180,200) # total number of time points (20/range(27) for ngram/nyt)
# emb_all = sio.loadmat('../results/emb_frobreg10_diffreg50_symmreg10_iter10.mat')

FileNotFoundError: [Errno 2] No such file or directory: '../results/emb_frobreg10_diffreg50_symmreg10_iter10.mat'

## Example embedding over time

In [40]:
target_word = 'communist'
plt.figure(figsize=(12, 8))

# Plot the trajectory line
plt.plot(traj[:, 0], traj[:, 1], linestyle='--', color='gray', alpha=0.5, zorder=1)

# Plot each time point
for i, (w_label, year) in enumerate(list_of_words):
    if isword[i]:
        plt.scatter(Z[i, 0], Z[i, 1], color='red', s=100, edgecolors='black', zorder=2)
        plt.text(Z[i, 0] + 0.2, Z[i, 1] + 0.2, f"{year*10}", fontsize=9, fontweight='bold')

plt.title(f"Semantic Drift Trajectory for '{target_word}' (1800s - 1990s)")
plt.xlabel("TSNE Dimension 1")
plt.ylabel("TSNE Dimension 2")
plt.grid(True, linestyle=':', alpha=0.6)

NameError: name 'traj' is not defined

<Figure size 1200x800 with 0 Axes>

## Semantic similarity heatmap over time

In [ ]:
# Extract embeddings for the target word across all time points
word_idx = word2Id['communist']
time_embeddings = []

for year in times:
    # Use index to match your loadmat keys
    t_idx = times.index(year)
    emb = emb_all[f'U_{t_idx}'][word_idx, :].reshape(1, -1)
    time_embeddings.append(emb[0])

# Calculate self-similarity matrix
sim_matrix = cosine_similarity(time_embeddings)

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, xticklabels=[t*10 for t in times], yticklabels=[t*10 for t in times],
            annot=False, cmap='YlGnBu')
plt.title("Word Self-Similarity Over Time (Cosine)")

## Closest words

In [ ]:
results = {}
for i, year in enumerate(times):
    # Get top 5 neighbors excluding the word itself
    emb = emb_all[f'U_{i}']
    # Re-normalize for dot product = cosine similarity
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb_n = emb / (norms + 1e-9)

    v = emb_n[word2Id['communist'], :]
    sims = np.dot(emb_n, v)
    nearest = np.argsort(sims)[-6:-1][::-1] # exclude self

    results[year*10] = [wordlist[idx] for idx in nearest]

df_neighbors = pd.DataFrame(results)
print("Top 5 Nearest Neighbors by Decade:")
display(df_neighbors)